In [5]:
import os
from terrain_classification.svm_classification.train_svm import SVMClassification
# from terrain_classification.svm_classification.predict_svm import PredictSVM
import numpy as np


In [6]:

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score


In [7]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), "terrain_classification"))
files_to_use = {
    # "sand": ["trial1.csv"],
    "sand": ["trial1.csv", "trial2.csv", "trial3.csv", "trial4.csv", "trial5.csv", "trial6.csv", "trial7.csv", "trial8.csv", ],
    "concrete": ["trial1.csv", "trial2.csv", "trial3.csv", "trial4.csv", "trial5.csv", "trial6.csv", "trial7.csv", "trial8.csv", ],
    "gravel": ["trial1.csv", "trial2.csv", "trial3.csv", "trial4.csv", "trial5.csv", "trial6.csv", "trial7.csv", "trial8.csv", ],
    # "sand": ["trial1.csv", "trial2.csv", "trial3.csv", "trial4.csv"],
    # "concrete": ["trial1.csv", "trial2.csv", "trial3.csv", "trial4.csv"],
    # "gravel": ["trial1.csv", "trial2.csv", "trial3.csv", "trial4.csv"],
}

data_labels = {}
for key_ in files_to_use.keys():
    for file_ in files_to_use[key_]:
        data_labels[os.path.join(parent_dir, 'data', key_, file_)] = key_

# data_labels

In [8]:
# Create an instance of the SVMClassification class
svm_classifier = SVMClassification(data_labels)

# Load the data
# cmp = ['x', 'y', 'z']
cmp = ['x', 'z']

svm_classifier.create_feature_matrix_and_label(normalize_data=False,
                            legs=['fl', 'fr', 'rl', 'rr'], 
                            components=cmp,
                            combine_components=True)

unique_labels, label_counts = np.unique(svm_classifier.labels, return_counts=True)
for label, count in zip(unique_labels, label_counts):
    print(f"Label {label}: {count} samples")

print(f"ratio {max(label_counts)/min(label_counts)}")
# print(f"step shape: {len(svm_classifier.data_extractor.steps)}")
# print(f"step size: {[len(s) for s in svm_classifier.data_extractor.steps.values()]}")
# # print(f"step size: {[len(i) for s in svm_classifier.data_extractor.steps.values() for i in range(len(s))]}")
# print(f'steps: {svm_classifier.data_extractor.steps.keys()}')
# print(f"feature matrix shape: {svm_classifier.feature_matrix.shape}")

Loading data from /Users/brolin/Documents/RWTH/terrain_property_prediction/src/terrain_classification/data/sand/trial1.csv with label sand
Loading data from /Users/brolin/Documents/RWTH/terrain_property_prediction/src/terrain_classification/data/sand/trial2.csv with label sand
Loading data from /Users/brolin/Documents/RWTH/terrain_property_prediction/src/terrain_classification/data/sand/trial3.csv with label sand
Loading data from /Users/brolin/Documents/RWTH/terrain_property_prediction/src/terrain_classification/data/sand/trial4.csv with label sand
Loading data from /Users/brolin/Documents/RWTH/terrain_property_prediction/src/terrain_classification/data/sand/trial5.csv with label sand
Loading data from /Users/brolin/Documents/RWTH/terrain_property_prediction/src/terrain_classification/data/sand/trial6.csv with label sand
Loading data from /Users/brolin/Documents/RWTH/terrain_property_prediction/src/terrain_classification/data/sand/trial7.csv with label sand
Loading data from /Users/br

In [9]:
#train with sklearn 
#using python 3.8
report_dir = os.path.join(parent_dir, "svm_classification")
svm_classifier.train_classifier(C=100, gamma=0.001, find_best_parameters=False, 
                                save_model=True, model_file_name="test", 
                                save_report=True, file_path=None)

print("Training completed.")
print("Report: \n")
# print(svm_classifier.report)
svm_classifier.print_classification_report()


[Training] Training SVM with fixed parameters...
Classification report saved to /Users/brolin/Documents/RWTH/terrain_property_prediction/src/terrain_classification/svm_classification/reports/test.json
Model saved to /Users/brolin/Documents/RWTH/terrain_property_prediction/src/terrain_classification/svm_classification/models/test.joblib
Training completed.
Report: 

              precision    recall  f1-score   support

    concrete       0.47      0.56      0.51       113
      gravel       0.49      0.46      0.47       107
        sand       0.89      0.79      0.84       125

    accuracy                           0.61       345
   macro avg       0.62      0.60      0.61       345
weighted avg       0.63      0.61      0.62       345



In [10]:

# #to train with cuML, we need to convert the labels to integers
# l_ = ['sand', 'gravel', 'concrete']
# min_len = [len(np.where(svm_classifier.labels == l)[0]) for l in l_ if len(np.where(svm_classifier.labels == l)[0]) > 0]

# print(f"min_len {min_len}")
# new_ids = np.empty(0)

# if use_sand:
#     new_ids = np.concatenate((new_ids, np.where(svm_classifier.labels == 'sand')[0]))
# if use_gravel:
#     new_ids = np.concatenate((new_ids, np.where(svm_classifier.labels == 'gravel')[0]))
# if use_concrete:
#     new_ids = np.concatenate((new_ids, np.where(svm_classifier.labels == 'concrete')[0]))

# new_ids = new_ids.astype(int)
# new_labels = svm_classifier.labels[new_ids]
# new_features = svm_classifier.feature_matrix[new_ids]

# print(f"new labels shape {new_labels.shape}")
# print(f"new features shape {new_features.shape}")


In [11]:
def convert_labels(labels):
    # Convert string labels to integers
    label_map = {}
    label_map['sand'] = 0
    label_map['gravel'] = 1
    label_map['concrete'] = 2
    return np.array([label_map[label] for label in labels], dtype=int)

In [12]:
#use miniconda python 3.12
new_X_train, new_X_test, new_y_train, new_y_test = train_test_split(svm_classifier.feature_matrix, svm_classifier.labels, test_size=0.2, random_state=42)
# new_X_train, new_X_test, new_y_train, new_y_test = train_test_split(new_features, new_labels, test_size=0.2, random_state=42)
# print(f"X type {new_X_train.dtype}")
# print(f"y type {new_y_train.dtype}")
orig_label_train = new_y_train
orig_label_test = new_y_test
new_y_train = convert_labels(new_y_train)
new_y_test = convert_labels(new_y_test)

scaler = StandardScaler()
new_X_train = scaler.fit_transform(new_X_train)
new_X_test = scaler.transform(new_X_test)


In [ ]:

from cuml.svm import SVC as cmSVC
from cuml.metrics import accuracy_score

# print("Creating SVC model")
# c = [0.1, 1, 10, 100]
c = [1000, 10, 100]
g = [0.001, 0.01, 0.1, 1]
# g = ['scale', 'auto']
# g = [0.01, 1]
kernel = ['rbf']
# kernel = ['rbf', 'poly', 'sigmoid']
c
# for c_, g_ in zip(c, g):
for k_ in kernel:
    for c_ in c:
        for g_ in g:
        # print(f"Trying c: {c_} g: {g_}")
        # print(f"accuracy {accuracy_score(new_y_pred, new_y_test)}")
            new_svc = cmSVC(kernel=k_, C=c_, gamma=g_, cache_size=2000)
            new_svc.fit(new_X_train, new_y_train)
            new_y_pred = new_svc.predict(new_X_test)
            print(f"Trying c: {c_} g: {g_} kernel: {k_} -> Accuracy score: {accuracy_score(new_y_pred, new_y_test):.4f}") 
            # print(classification_report(new_y_test, new_y_pred))
            # print("=========================")
        print("----------------------")
    print("----------------------")

print("All Done")

In [ ]:
from joblib import dump
from cuml.svm import SVC as cmSVC

# from sklearn.externals import joblib

new_svc = cmSVC(kernel='rbf', C=100, gamma=0.01, cache_size=2000)
new_svc.fit(new_X_train, new_y_train)
print("Training done")
# sk_model = new_svc.to_sklearn()
# dump(sk_model, os.path.join(parent_dir, 'svm_classification/cpu_test_model.joblib'))

Training done


In [ ]:
from joblib import load
# import cuml
from sklearn.metrics import classification_report
loaded_model = load(os.path.join(parent_dir, 'svm_classification/models/trained_test_model.joblib'))
print(f"loaded model {loaded_model}")

predictions_ = loaded_model.predict(new_X_train)
print(classification_report(orig_label_train, predictions_))
# print(f"Accuracy: {accuracy_score(new_y_test, predictions_):.4f}")

loaded model SVC(C=10, gamma=0.01)
              precision    recall  f1-score   support

    concrete       0.96      0.97      0.97       951
      gravel       0.96      0.97      0.96       942
        sand       1.00      0.99      0.99       869

    accuracy                           0.97      2762
   macro avg       0.97      0.97      0.97      2762
weighted avg       0.97      0.97      0.97      2762

